### Parse Surveys

In [55]:
import ast
post_surveys = []
pre_surveys = []
users = set()
with open("joint_survey.txt", "r") as infile:
    for line in infile:
        user, survey = line.split("||")
        user = user.split(":")[1].strip()
        users.add(user)
        survey = survey[13:].strip()
        survey = ast.literal_eval(survey)
        survey["user"] = user
        if "POST-SURVEY" in line:
            post_surveys.append(survey)
        else:
            pre_surveys.append(survey)   

### Get Real Objective Metrics From Filtered Transcript

In [18]:
def parse_transcript(transcipt_name):

    chat_stats = {}
    policy = None
    dialogs = {}
    DIALOG_TYPES = ['hdc', 'faq', 'cts_llm']

    user_dialog_nums = {}

    with open(transcipt_name, "r") as transcript:
        for line in transcript:
            if "(POLICY:" in line:
                current_dialog = {}
                tmp = line.split()
                user = tmp[1].strip()
                policy = tmp[3].strip(")").strip()
                current_dialog['user'] = user
                current_dialog['turns'] = []
                # TODO: add back goals to dialogs and analyze if there is anything interesting at a per/goal level
                # goal_type = tmp[-1].strip(")").strip()
            elif "USER:" in line and not "POST-NLU" in line:
                current_dialog["turns"].append(line)
            elif "SYSTEM" in line:
                current_dialog['turns'].append(line)
            elif "DIALOG END:" in line:
                current_dialog["end_condition"] = line.split(":")[1].strip()
            elif "SUBJECTIVE LENGTH" in line:
                current_dialog["sub_length"] = line.split(":")[1].strip()
            elif "SUBJECTIVE QUALITY" in line:
                current_dialog["sub_quality"] = line.split(":")[1].strip()
            elif line.strip() == "":
                # TODO: Change this line to analyse one group at a time
                if current_dialog and policy in DIALOG_TYPES:
                    obj_length = len(current_dialog["turns"])
                    if not policy in chat_stats:
                        chat_stats[policy] = {"length": [], "end_condition": [], "sub_length": [], "sub_quality": []}
                    chat_stats[policy]["length"].append(obj_length)
                    chat_stats[policy]["end_condition"].append(current_dialog["end_condition"])
                    chat_stats[policy]["sub_length"].append(int(current_dialog["sub_length"]))
                    chat_stats[policy]["sub_quality"].append(int(current_dialog["sub_quality"]))
                    current_dialog["length"] = obj_length
                    if current_dialog['user'] not in user_dialog_nums:
                        user_dialog_nums[current_dialog['user']] = 0
                    user_dialog_nums[current_dialog['user']] += 1
                    if policy not in dialogs:
                        dialogs[policy] = []
                    dialogs[policy].append(current_dialog)
                    current_dialog = {}
    return (chat_stats, dialogs, user_dialog_nums)

In [185]:
chat_stats, dialogs, user_dialog_nums = parse_transcript(transcipt_name='joint_transcript-filtered_new.txt')

### Collect corpus statistics per/policy
* Number 
* Length
* #Success
* Avg. first utterance length
* Avg. all user utterance lengths

In [183]:
from scipy.stats import f_oneway
from scipy.stats import tukey_hsd
def calc_dialog_metrics(chat_stats, dialogs):
    num_dialogs = {}
    success = {}
    avg_len_dialog = {}
    avg_len_first_utterance = {}
    avg_len_all_utterances = {}
    avg_sub_quality = {}
    avg_sub_len = {}

    for policy in chat_stats:
        num_dialogs[policy] = 0
        success[policy] = 0
        avg_len_dialog[policy] = 0
        avg_len_first_utterance[policy] = 0
        avg_len_all_utterances[policy] = 0
        avg_sub_quality[policy] = 0
        avg_sub_len[policy] = 0
        
        success[policy] = chat_stats[policy]["end_condition"].count("SUCCESS") + chat_stats[policy]["end_condition"].count("SUCCESS - OTHER QUESTION")
        avg_len_dialog[policy] = sum(chat_stats[policy]["length"])
        avg_sub_quality[policy] = sum(chat_stats[policy]["sub_quality"])
        avg_sub_len[policy] = sum(chat_stats[policy]["sub_length"])

        for record in dialogs[policy]:
            num_dialogs[policy] += 1
            d = record['turns']
            user_turns = [t for t in d if "USER" in t]
            len_user_turns = [len(t[6:].split()) for t in user_turns]
            avg_len_first_utterance[policy] += len_user_turns[0]
            avg_len_all_utterances[policy] += sum(len_user_turns)/len(len_user_turns)

        avg_len_dialog[policy] = avg_len_dialog[policy]/num_dialogs[policy]
        avg_sub_len[policy] = avg_sub_len[policy]/num_dialogs[policy]
        avg_sub_quality[policy] = avg_sub_quality[policy]/num_dialogs[policy]

        avg_len_first_utterance[policy] = avg_len_first_utterance[policy]/num_dialogs[policy]
        avg_len_all_utterances[policy] = avg_len_all_utterances[policy]/num_dialogs[policy]

        print(policy)
        print(f"NUM DIALOGS: {num_dialogs[policy]}")
        print(f"COUNT SUCCESS: {success[policy]}")
        print(f"PERCENT SUCCESS: {success[policy]/num_dialogs[policy]*100}")
        print(f"SUBJECTIVE QUALITY: {avg_sub_quality[policy]}")
        print(f"AVG NUM TURNS: {avg_len_dialog[policy]}")
        print(f"SUBJECTIVE LENGTH: {avg_sub_len[policy]}")

        print(f"AVG LEN INITIAL UTTERANCE: {avg_len_first_utterance[policy]}")
        print(f"AVG LEN ALL UTTERANCES: {avg_len_all_utterances[policy]}")
    
    print("length")
    print(f_oneway(chat_stats["cts_llm"]["length"], chat_stats["hdc"]["length"], chat_stats["faq"]["length"]))
    print(tukey_hsd(chat_stats["cts_llm"]["length"], chat_stats["hdc"]["length"], chat_stats["faq"]["length"]))

In [186]:
calc_dialog_metrics(chat_stats=chat_stats, dialogs=dialogs)

cts_llm
NUM DIALOGS: 62
COUNT SUCCESS: 50
PERCENT SUCCESS: 80.64516129032258
SUBJECTIVE QUALITY: 2.935483870967742
AVG NUM TURNS: 10.612903225806452
SUBJECTIVE LENGTH: 2.8870967741935485
AVG LEN INITIAL UTTERANCE: 11.806451612903226
AVG LEN ALL UTTERANCES: 8.960304659498208
faq
NUM DIALOGS: 62
COUNT SUCCESS: 35
PERCENT SUCCESS: 56.451612903225815
SUBJECTIVE QUALITY: 2.596774193548387
AVG NUM TURNS: 2.2580645161290325
SUBJECTIVE LENGTH: 2.2903225806451615
AVG LEN INITIAL UTTERANCE: 10.209677419354838
AVG LEN ALL UTTERANCES: 10.293010752688174
hdc
NUM DIALOGS: 56
COUNT SUCCESS: 26
PERCENT SUCCESS: 46.42857142857143
SUBJECTIVE QUALITY: 2.857142857142857
AVG NUM TURNS: 15.053571428571429
SUBJECTIVE LENGTH: 2.732142857142857
AVG LEN INITIAL UTTERANCE: 9.767857142857142
AVG LEN ALL UTTERANCES: 5.761643862090291
length
F_onewayResult(statistic=32.113754640335316, pvalue=1.2614190348874386e-12)
Tukey's HSD Pairwise Group Comparisons (95.0% Confidence Interval)
Comparison  Statistic  p-value  L

### Parse out Trust and Usability scores

In [264]:
import ast
import csv

def parse_surveys(survey_file, outfile, user_dialog_nums):
    post_surveys = []
    pre_surveys = []
    users = set()
    with open(survey_file, "r") as infile:
        for line in infile:
            if "PREFERRED_STYLE" not in line:
                user, survey = line.split("||")
                user = user.split(":")[1].strip()
                users.add(user)
                survey = survey[13:].strip()
                survey = ast.literal_eval(survey)
                survey["user"] = user
                if "POST-SURVEY" in line:
                    post_surveys.append(survey)
                else:
                    pre_surveys.append(survey)   

    # Remove Users who don't interact with the system
    user_black_list = set()
    user_gray_list = set()

    for user in users:
        if user in user_dialog_nums:
            if user_dialog_nums[user] != 3:
                user_gray_list.add(user)
        else:
            user_black_list.add(user)

    print(f"Removed {len(user_black_list)} users: {user_black_list}")
    print("To Investigate: ", user_gray_list)
    users = [user for user in users if user not in user_black_list]
    print(len(users))

    if not outfile is None:
        # Save surveys to CSV for easier conent analysis
        with open("pre_survey.csv", "w", newline='') as outfile:
            fieldnames = pre_surveys[0].keys()
            writer = csv.DictWriter(outfile, fieldnames=fieldnames, delimiter="|")
            writer.writeheader()
            for s in pre_surveys:
                if s['user'] in user_black_list:
                    continue
                writer.writerow(s)

        with open("post_survey.csv", "w", newline='') as outfile:
            fieldnames = post_surveys[0].keys()
            writer = csv.DictWriter(outfile, fieldnames=fieldnames, delimiter="|")
            writer.writeheader()
            for s in post_surveys:
                if s['user'] in user_black_list:
                    continue
                writer.writerow(s)

    return pre_surveys, post_surveys, user_black_list

In [266]:
pre_surveys, post_surveys, user_black_list = parse_surveys(survey_file='joint_survey.txt', outfile=None, user_dialog_nums=user_dialog_nums)

Removed 27 users: {'1b9145d1d36ee5a4af1fd329d22168', 'e5e122c5a2ead5afc06e4df4c8cf78', 'ed751396034b4e87fb50846540d2b2', '5a465be12adc3998b4fe940817cddc', 'bf4b5adecf96a4d3eccdcf42e32d91', '92c512341aec50139fc63dab2be508', 'efe3e9656f07991690a7ef1d71908b', '63fe6663ab1740ce3d095bd3ec270f', '232cd8e82603855aebc8caa4d53fc1', '6e6ed1b576c8f271b519a059d5bf0d', '3531297f0bd32d54dcc23e8e8a50ad', '529f300b63822660867384ecba74e3', 'c951b710a16924c54f90bccc29460b', 'f2cb60fd5010d40e8c6d0b5a7ba1ae', 'cb723f719b2f5633ede9aa7ae5926c', '47c97ed8e9ba2195e48f663a9f35a7', '80dce4aa2bb6e4747dea90c91ca9fe', 'a3b3dff3fa294af607a57ff19c591d', '954201158613ce9098104a01786b76', '0f97158978beebaeb59f4dda0db17c', '3d9670b634a7a64e7a91bbd34c1823', 'e5cb9660f35b9a87793344d8882510', 'd3f5d2249e51c387c882cb587ba9ca', '5bbe9193fb7d45e2e10d3aade3aa89', 'e100425dee079289c5660a0b2db5d1', 'ba401ddeae9aef6345d8c3dbddb3bf', 'bf83f3984f247796d4437afe55ffa1'}
To Investigate:  {'85452be9505ef3ee3499d59b5300f3', '13bec8a37d

### Save data to csv format

In [8]:
# import csv
# with open("pre_survey_cts.csv", "w", newline='') as outfile:
#     fieldnames = pre_surveys[0].keys()
#     writer = csv.DictWriter(outfile, fieldnames=fieldnames, delimiter="|")
#     writer.writeheader()
#     for s in pre_surveys:
#         if s['user'] in user_black_list:
#             continue
#         writer.writerow(s)

# with open("post_surveys_cts.csv", "w", newline='') as outfile:
#     fieldnames = post_surveys[0].keys()
#     # TODO: expectations_2 should be 'likes' expectations_3 should be 'dislikes'
#     writer = csv.DictWriter(outfile, fieldnames=fieldnames)
#     writer.writeheader()
#     for s in post_surveys:
#         if s['user'] in user_black_list:
#             continue
#         writer.writerow(s)

ValueError: dict contains fields not in fieldnames: 'likes', 'dislikes'

In [66]:
import numpy as np

def parse_trust_and_usability_scores(post_surveys, user_black_list):
    trust = []
    reliability = []
    usability = []

    u_usability = {}
    u_trust = {}
    u_reliability = {}

    for res in post_surveys:
        user = res["user"]
        if user in user_black_list or user == "47e68725c26f72d805709141e76fd0" or user == "57affbcf1a53cf8152a4f84b337572":
            continue
        user_reliability = (int(res["reliability_1"]) + int(res["reliability_2"]) + (6 - int(res["reliability_3"])) + int(res["reliability_4"]) + (6 - int(res["reliability_5"])) + int(res["reliability_6"])) / 6
        user_trust = (int(res["trust_1"]) + int(res["trust_2"])) / 2
        # should be 0 to 4 scale, not 1 to 5
        user_usability = ((int(res['umux_1']) - 1) + (5 - int(res['umux_2'])) + (int(res['umux_3']) - 1) + (5 - int(res['umux_4']))) / 16 * 100
        u_usability[user] = user_usability
        trust.append(user_trust)
        u_trust[user] = user_trust
        reliability.append(user_reliability)
        u_reliability[user] = user_reliability
        usability.append(user_usability)
        
        # print(f" USER: {user}: Trust: {user_trust} Reliability: {user_reliability} Usability: {user_usability}")
        
    print(f"TRUST: {np.mean(trust)} +/- {np.std(trust)}")
    print(f"RELIABILITY: {np.mean(reliability)} +/- {np.std(reliability)}")
    print(f"USABILITY: {np.mean(usability)} +/- {np.std(usability)}")
    return trust, reliability, usability

In [188]:
u_usability = {}
u_trust = {}
u_reliability = {}

for res in post_surveys:
    user = res["user"]
    if user in user_black_list or user == "47e68725c26f72d805709141e76fd0" or user == "57affbcf1a53cf8152a4f84b337572":
        continue
    user_reliability = (int(res["reliability_1"]) + int(res["reliability_2"]) + (6 - int(res["reliability_3"])) + int(res["reliability_4"]) + (6 - int(res["reliability_5"])) + int(res["reliability_6"])) / 6
    user_trust = (int(res["trust_1"]) + int(res["trust_2"])) / 2
    # should be 0 to 4 scale, not 1 to 5
    user_usability = ((int(res['umux_1']) - 1) + (5 - int(res['umux_2'])) + (int(res['umux_3']) - 1) + (5 - int(res['umux_4']))) / 16 * 100
    u_usability[user] = user_usability
    u_trust[user] = user_trust
    u_reliability[user] = user_reliability

In [153]:
def create_user_condition_mapping(user_file):
    user_condition_mapping = {}
    with open(user_file, "r") as infile:
        for line in infile:
            if "GROUP" in line:
                user, group = line.split("||")
                user = user.split(":")[1].strip()
                group = group.split(":")[1].strip()
                user_condition_mapping[user] = group
    return user_condition_mapping


user_condition_mapping = create_user_condition_mapping("joint_user_log.txt")

In [69]:
def scores_per_condition(user_condition_mapping, surveys, blacklist):
    condition_surveys = {}
    for survey in surveys:
        user = survey["user"]
        condition = user_condition_mapping[user]
        if condition not in condition_surveys:
            condition_surveys[condition] = []
        condition_surveys[condition].append(survey)

    assert(len(condition_surveys) == 3)

    for condition in condition_surveys:
        print(condition)
        parse_trust_and_usability_scores(condition_surveys[condition], blacklist)

In [97]:
from scipy.stats import mannwhitneyu
from scipy.stats import tukey_hsd
from scipy.stats import f_oneway

def stats_per_condition(user_condition_mapping, surveys, blacklist):
    condition_surveys = {}

    for survey in surveys:
        user = survey["user"]
        if user in blacklist:
            continue
        condition = user_condition_mapping[user]
        if condition not in condition_surveys:
            condition_surveys[condition] = []
        condition_surveys[condition].append(survey)

    trust_dict = {}
    reliability_dict = {}
    usability_dict = {}
    for condition in condition_surveys:
        print(condition)
        trust, reliability, usability = parse_trust_and_usability_scores(condition_surveys[condition], blacklist)
        trust_dict[condition] = trust
        reliability_dict[condition] = reliability
        usability_dict[condition] = usability
        
    print("\n")
    print("TRUST")
    print(f_oneway(trust_dict["cts_llm"], trust_dict["hdc"], trust_dict["faq"]))
    print(tukey_hsd(trust_dict["cts_llm"], trust_dict["hdc"], trust_dict["faq"]))
    print("RELIABILITY")
    print(f_oneway(reliability_dict["cts_llm"], reliability_dict["hdc"], reliability_dict["faq"]))
    print(tukey_hsd(reliability_dict["cts_llm"], reliability_dict["hdc"], reliability_dict["faq"]))
    print("USABILITY")
    print(f_oneway(usability_dict["cts_llm"], usability_dict["hdc"], usability_dict["faq"]))
    print(tukey_hsd(usability_dict["cts_llm"], usability_dict["hdc"], usability_dict["faq"]))    
    # print("ADAPTIVE VS HDC")
    # print("trust", mannwhitneyu(trust_dict["cts_llm"], trust_dict["hdc"]))
    # print("reliability", mannwhitneyu(reliability_dict["cts_llm"], reliability_dict["hdc"]))
    # print("usability", mannwhitneyu(usability_dict["cts_llm"], usability_dict["hdc"]))
    # print("\n")
    # print("ADAPTIVE VS FAQ")
    # print("trust", mannwhitneyu(trust_dict["cts_llm"], trust_dict["faq"]))
    # print("reliability", mannwhitneyu(reliability_dict["cts_llm"], reliability_dict["faq"]))
    # print("usability", mannwhitneyu(usability_dict["cts_llm"], usability_dict["faq"]))

In [189]:
stats_per_condition(user_condition_mapping=user_condition_mapping, surveys=post_surveys, blacklist=user_black_list)

faq
TRUST: 2.8333333333333335 +/- 0.8637312927246217
RELIABILITY: 2.7936507936507935 +/- 0.6917976772171716
USABILITY: 57.73809523809524 +/- 21.469867615323132
cts_llm
TRUST: 3.380952380952381 +/- 1.153718232341497
RELIABILITY: 3.1984126984126986 +/- 0.8028043565969393
USABILITY: 66.66666666666667 +/- 22.68552261923751
hdc
TRUST: 3.15 +/- 1.3143439428094916
RELIABILITY: 3.0583333333333336 +/- 0.9578259526425223
USABILITY: 58.4375 +/- 29.86133317100896


TRUST
F_onewayResult(statistic=1.1977116926272473, pvalue=0.3091166399601872)
Tukey's HSD Pairwise Group Comparisons (95.0% Confidence Interval)
Comparison  Statistic  p-value  Lower CI  Upper CI
 (0 - 1)      0.231     0.797    -0.634     1.096
 (0 - 2)      0.548     0.279    -0.306     1.402
 (1 - 0)     -0.231     0.797    -1.096     0.634
 (1 - 2)      0.317     0.655    -0.548     1.181
 (2 - 0)     -0.548     0.279    -1.402     0.306
 (2 - 1)     -0.317     0.655    -1.181     0.548

RELIABILITY
F_onewayResult(statistic=1.247180

### What mental models did users have?

In [262]:
import plotly.graph_objects as go
from scipy import stats


mental_models = {
      "natural language": [],
      "keywords": [],
      "specific question": [],
      "follow-up questions": [],
      "general answer": [],
      "personalized answer": [],
      "immediate answer": [],
      "long dialog": []
}

user_mms = {}


for res in pre_surveys:
    user = res["user"]
    if user not in user_black_list:
        mental_models["natural language"].append(int(res["chat_exp_1"]))
        mental_models["keywords"].append(int(res["chat_exp_2"]))
        mental_models["specific question"].append(int(res["chat_exp_3"]))
        mental_models["follow-up questions"].append(int(res['chat_exp_4']))
        mental_models["general answer"].append(int(res["chat_exp_5"]))
        mental_models["personalized answer"].append(int(res["chat_exp_6"]))
        mental_models["immediate answer"].append(int(res['chat_exp_7']))
        mental_models["long dialog"].append(int(res["chat_exp_8"]))
        user_mms[user] = {key: mental_models[key][-1] for key in mental_models}
    else:
        print(user)
    
labels = [key for key in mental_models]
avg_mental_models = [np.mean(mental_models[l]) for l in labels]
yes_count = []
no_count = []
for l in labels:
    ys = [entry for entry in mental_models[l] if entry > 3]
    ns = [entry for entry in mental_models[l] if entry <= 3]
    yes_count.append(len(ys))
    no_count.append(len(ns))
print(mental_models)
print(avg_mental_models)

fig = go.Figure()
fig.add_trace(go.Bar(
    name='Expect',
    x=labels,
    y=yes_count
))
fig.add_trace(go.Bar(
    name="Do Not Expect",
    x=labels,
    y=no_count
))
# fig.add_trace(go.Bar(
#     name='Control',
#     x=labels, y=avg_mental_models,
#     error_y=dict(type='data', array=[stats.sem(mental_models[l]) for l in labels])
# ))
fig.update_layout(
    barmode='group', 
    width=600,
    yaxis_title="# Users",
    plot_bgcolor = 'rgba(0,0,0,0)',
    legend={"orientation": "h", "yanchor": "top", "y":1.12, "xanchor":"center", "x":0.5})
fig.update_xaxes(showline=True, linewidth=2, linecolor='darkgrey', gridcolor='rgba(0,0,0,0)')
fig.update_yaxes(showline=False, linewidth=2, linecolor='darkgrey', gridcolor='darkgrey')
fig.show()
fig.write_image("llm_pre_mms.pdf")

80dce4aa2bb6e4747dea90c91ca9fe
c951b710a16924c54f90bccc29460b
3531297f0bd32d54dcc23e8e8a50ad
d3f5d2249e51c387c882cb587ba9ca
47c97ed8e9ba2195e48f663a9f35a7
5bbe9193fb7d45e2e10d3aade3aa89
e100425dee079289c5660a0b2db5d1
ed751396034b4e87fb50846540d2b2
0f97158978beebaeb59f4dda0db17c
92c512341aec50139fc63dab2be508
e5e122c5a2ead5afc06e4df4c8cf78
63fe6663ab1740ce3d095bd3ec270f
5a465be12adc3998b4fe940817cddc
3d9670b634a7a64e7a91bbd34c1823
6e6ed1b576c8f271b519a059d5bf0d
efe3e9656f07991690a7ef1d71908b
a3b3dff3fa294af607a57ff19c591d
bf4b5adecf96a4d3eccdcf42e32d91
ba401ddeae9aef6345d8c3dbddb3bf
bf83f3984f247796d4437afe55ffa1
e5cb9660f35b9a87793344d8882510
cb723f719b2f5633ede9aa7ae5926c
232cd8e82603855aebc8caa4d53fc1
f2cb60fd5010d40e8c6d0b5a7ba1ae
529f300b63822660867384ecba74e3
954201158613ce9098104a01786b76
1b9145d1d36ee5a4af1fd329d22168
{'natural language': [4, 5, 4, 4, 4, 3, 3, 2, 4, 1, 5, 4, 5, 3, 4, 2, 2, 5, 2, 2, 4, 3, 4, 3, 2, 3, 4, 4, 5, 4, 4, 5, 4, 5, 2, 5, 5, 2, 4, 4, 3, 2, 5, 5, 5, 4, 5, 

### What role does Mental model have on dialog length?

In [279]:
from scipy.stats import ttest_ind
from collections import defaultdict
mms_by_agent = {}
for condition in dialogs:
    for d in dialogs[condition]:
        user = d["user"]
        if user in user_black_list:
            continue
        if condition not in mms_by_agent:
            mms_by_agent[condition] = {}
        for mm in user_mms[user]:
            if mm not in mms_by_agent[condition]:
                mms_by_agent[condition][mm] = {"yes": [], "no": []}
            if user_mms[user][mm] > 3:
                mms_by_agent[condition][mm]["yes"].append(d["length"])
            elif user_mms[user][mm]<= 3:
                mms_by_agent[condition][mm]["no"].append(d["length"])

yesses = defaultdict(lambda: list())
nos = defaultdict(lambda: list())
for condition in mms_by_agent:
    print(condition)
    for mm in mms_by_agent[condition]:
        print(mm)
        yesses[mm] += mms_by_agent[condition][mm]["yes"]
        nos[mm] += mms_by_agent[condition][mm]["no"]
        print(np.mean(mms_by_agent[condition][mm]["yes"]) - np.mean(mms_by_agent[condition][mm]["no"]))
        print(mms_by_agent[condition][mm]["no"])
        print(ttest_ind(mms_by_agent[condition][mm]["yes"], mms_by_agent[condition][mm]["no"]))
    print('\n')
print("combined")
for mm in yesses:
    print(mm)
    print(ttest_ind(yesses[mm], nos[mm]))

cts_llm
natural language
-2.343366778149388
[3, 5, 8, 3, 4, 61, 4, 4, 11, 5, 8, 7, 5, 7, 8, 29, 8, 14, 5, 5, 22, 31, 21]
Ttest_indResult(statistic=-0.7869518059727141, pvalue=0.43440756523891233)
keywords
4.694179894179894
[18, 5, 4, 24, 5, 13, 26, 3, 4, 11, 5, 11, 8, 3, 5, 7, 5, 7, 3, 5, 4, 5, 5, 9, 8, 8, 4]
Ttest_indResult(statistic=1.6456754007054188, pvalue=0.10505974058693311)
specific question
0.8636363636363633
[61, 4, 4, 26, 3, 4, 11, 5, 11, 8, 3, 5, 7, 5, 4, 5, 5, 9]
Ttest_indResult(statistic=0.2712948537638408, pvalue=0.7870954916887196)
follow-up questions
1.694915254237289
[11, 5, 11]
Ttest_indResult(statistic=0.251678188896952, pvalue=0.8021498706196137)
general answer
4.497354497354497
[8, 3, 4, 61, 4, 4, 11, 5, 11, 8, 3, 5, 7, 5, 4, 3, 5, 4, 5, 5, 9, 8, 8, 4, 14, 5, 5]
Ttest_indResult(statistic=1.5737594256031984, pvalue=0.1208018461128391)
personalized answer
0.7243589743589745
[3, 5, 30, 51, 11, 8, 3, 4, 11, 5, 8, 18, 5, 4, 6, 5, 4, 7, 5, 7, 8, 29, 8, 8, 8, 4]
Ttest_in

### What role do expectations have on success?

In [281]:
from scipy.stats import barnard_exact
mms_by_agent = {}
for condition in dialogs:
    for d in dialogs[condition]:
        user = d["user"]
        if user in user_black_list:
            continue
        if condition not in mms_by_agent:
            mms_by_agent[condition] = {}
        for mm in user_mms[user]:
            if mm not in mms_by_agent[condition]:
                mms_by_agent[condition][mm] = {"yes": [], "no": []}
            if user_mms[user][mm] > 3:
                mms_by_agent[condition][mm]["yes"].append("SUCCESS" in d["end_condition"])
            elif user_mms[user][mm]<= 3:
                mms_by_agent[condition][mm]["no"].append("SUCCESS" in d["end_condition"])


yes_successes = defaultdict(lambda: 0)
yes_failures = defaultdict(lambda: 0)
no_successes = defaultdict(lambda: 0)
no_failures = defaultdict(lambda: 0)
for condition in mms_by_agent:
    print(condition)
    for mm in mms_by_agent[condition]:
        print(mm)
        yes_success = sum(mms_by_agent[condition][mm]["yes"])
        yes_failure = len(mms_by_agent[condition][mm]["yes"]) - yes_success
        no_success = sum(mms_by_agent[condition][mm]["no"])
        no_failure = len(mms_by_agent[condition][mm]["no"]) - no_success
        yes_successes[mm] += yes_success
        yes_failures[mm] += yes_failure
        no_successes[mm] += no_success
        no_failures[mm] += no_failure
        table = [[yes_success, no_success], [yes_failure, no_failure]]
        if no_failure > 0 or no_success > 0:
            print(yes_success/(yes_success+yes_failure) - no_success/(no_success+no_failure))
        # else:
        print(table)
        print(barnard_exact(table))
    print('\n')
print("combined")
for mm in yes_failures:
    print(mm)
    table = [[yes_successes[mm], no_successes[mm]], [yes_failures[mm], no_failures[mm]]]
    print(barnard_exact(table))

cts_llm
natural language
0.2452619843924192
[[35, 15], [4, 8]]
BarnardExactResult(statistic=2.361275597042341, pvalue=0.0199583980941692)
keywords
-0.14603174603174596
[[26, 24], [9, 3]]
BarnardExactResult(statistic=-1.443056720441957, pvalue=0.17510052916019817)
specific question
0.11868686868686873
[[37, 13], [7, 5]]
BarnardExactResult(statistic=1.0737062530530026, pvalue=0.3245514056777135)
follow-up questions
-0.2033898305084746
[[47, 3], [12, 0]]
BarnardExactResult(statistic=-0.8698334147936175, pvalue=0.5427375056981543)
general answer
0.24761904761904763
[[32, 18], [3, 9]]
BarnardExactResult(statistic=2.446922265097233, pvalue=0.01545493845826264)
personalized answer
-0.13461538461538458
[[27, 23], [9, 3]]
BarnardExactResult(statistic=-1.3238928611078415, pvalue=0.19344826694465514)
immediate answer
0.09625668449197855
[[42, 8], [9, 3]]
BarnardExactResult(statistic=0.7328794576635641, pvalue=0.5275185973179832)
long dialog
-0.09625668449197855
[[8, 42], [3, 9]]
BarnardExactResul

In [144]:
print(barnard_exact([[13, 16], [6, 24]]))

BarnardExactResult(statistic=2.040390696013265, pvalue=0.04641910400075573)


### Role of Mental Models on Perceived length

In [288]:
from scipy.stats import ttest_ind
mms_by_agent = {}
for condition in dialogs:
    for d in dialogs[condition]:
        user = d["user"]
        if user in user_black_list:
            continue
        if condition not in mms_by_agent:
            mms_by_agent[condition] = {}
        for mm in user_mms[user]:
            if mm not in mms_by_agent[condition]:
                mms_by_agent[condition][mm] = {"yes": [], "no": []}
            if user_mms[user][mm] > 3:
                mms_by_agent[condition][mm]["yes"].append(int(d["sub_length"]))
            elif user_mms[user][mm]<= 3:
                mms_by_agent[condition][mm]["no"].append(int(d["sub_length"]))

yesses = defaultdict(lambda: list())
nos = defaultdict(lambda: list())
for condition in mms_by_agent:
    print(condition)
    for mm in mms_by_agent[condition]:
        print(mm)
        yesses[mm] += mms_by_agent[condition][mm]["yes"]
        nos[mm] += mms_by_agent[condition][mm]["no"]
        print(np.mean(mms_by_agent[condition][mm]["yes"]), np.mean(mms_by_agent[condition][mm]["no"]))
        print(ttest_ind(mms_by_agent[condition][mm]["yes"], mms_by_agent[condition][mm]["no"]))
    print('\n')
print("combined")
for mm in yesses:
    print(mm)
    print(ttest_ind(yesses[mm], nos[mm]))

cts_llm
natural language
2.923076923076923 2.8260869565217392
Ttest_indResult(statistic=0.44055292105496585, pvalue=0.6611198935406757)
keywords
2.7142857142857144 3.111111111111111
Ttest_indResult(statistic=-1.9019611977094117, pvalue=0.061979968677705555)
specific question
2.8181818181818183 3.0555555555555554
Ttest_indResult(statistic=-1.020243531961008, pvalue=0.31170920469300967)
follow-up questions
2.8813559322033897 3.0
Ttest_indResult(statistic=-0.23911854521144768, pvalue=0.8118285750034471)
general answer
2.857142857142857 2.925925925925926
Ttest_indResult(statistic=-0.3204368908851863, pvalue=0.7497504976948497)
personalized answer
2.888888888888889 2.8846153846153846
Ttest_indResult(statistic=0.019796902729800225, pvalue=0.984271076716142)
immediate answer
2.9411764705882355 2.6363636363636362
Ttest_indResult(statistic=1.1042236205512042, pvalue=0.27390489112046834)
long dialog
2.1818181818181817 3.0392156862745097
Ttest_indResult(statistic=-3.350234272097525, pvalue=0.0014

/var/folders/xn/1drxn9fx2dxc6_bh03c675ym0000gq/T/ipykernel_69529/2328180470.py:27: RuntimeWarning:

Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.



### Role of Mental Models of Perceived Quality

In [284]:
from scipy.stats import ttest_ind
mms_by_agent = {}
for condition in dialogs:
    for d in dialogs[condition]:
        user = d["user"]
        if user in user_black_list:
            continue
        if condition not in mms_by_agent:
            mms_by_agent[condition] = {}
        for mm in user_mms[user]:
            if mm not in mms_by_agent[condition]:
                mms_by_agent[condition][mm] = {"yes": [], "no": []}
            if user_mms[user][mm] > 3:
                mms_by_agent[condition][mm]["yes"].append(int(d["sub_quality"]))
            elif user_mms[user][mm]<= 3:
                mms_by_agent[condition][mm]["no"].append(int(d["sub_quality"]))

yesses = defaultdict(lambda: list())
nos = defaultdict(lambda: list())
for condition in mms_by_agent:
    print(condition)
    for mm in mms_by_agent[condition]:
        print(mm)
        yesses[mm] += mms_by_agent[condition][mm]["yes"]
        nos[mm] += mms_by_agent[condition][mm]["no"]
        print(np.mean(mms_by_agent[condition][mm]["yes"]) - np.mean(mms_by_agent[condition][mm]["no"]))
        print(ttest_ind(mms_by_agent[condition][mm]["yes"], mms_by_agent[condition][mm]["no"]))
    print('\n')
print("combined")
for mm in yesses:
    print(mm)
    print(ttest_ind(yesses[mm], nos[mm]))

cts_llm
natural language
0.03567447045707928
Ttest_indResult(statistic=0.1521483650097264, pvalue=0.8795803874014654)
keywords
0.14814814814814836
Ttest_indResult(statistic=0.6506820243207851, pvalue=0.5177358562286354)
specific question
-0.012626262626262985
Ttest_indResult(statistic=-0.05059137018892682, pvalue=0.9598191877037139)
follow-up questions
-0.4180790960451981
Ttest_indResult(statistic=-0.7960794598675858, pvalue=0.4291243676105453)
general answer
0.2137566137566136
Ttest_indResult(statistic=0.9424453785469293, pvalue=0.3497440242115657)
personalized answer
0.15384615384615374
Ttest_indResult(statistic=0.6726446127275677, pvalue=0.5037557948428605)
immediate answer
0.36363636363636376
Ttest_indResult(statistic=1.2419093001010257, pvalue=0.21910272043116977)
long dialog
-0.2531194295900181
Ttest_indResult(statistic=-0.8587952204959711, pvalue=0.3938715209361431)


faq
natural language
-0.23076923076923084
Ttest_indResult(statistic=-1.0227575394375306, pvalue=0.31052856945073

### Role of Mental Models on Usability

In [287]:
from scipy.stats import ttest_ind
import numpy as np
mms_by_agent = {}

for user in user_mms:
    condition = user_condition_mapping[user]
    if user in user_black_list:
        continue
    if condition not in mms_by_agent:
        mms_by_agent[condition] = {}
    for mm in user_mms[user]:
        if mm not in mms_by_agent[condition]:
            mms_by_agent[condition][mm] = {"yes": [], "no": []}
        if user_mms[user][mm] > 3:
            mms_by_agent[condition][mm]["yes"].append(u_usability[user])
        elif user_mms[user][mm]<= 3:
            mms_by_agent[condition][mm]["no"].append(u_usability[user])
    
yesses = defaultdict(lambda: list())
nos = defaultdict(lambda: list())
for condition in mms_by_agent:
    print(condition)
    for mm in mms_by_agent[condition]:
        print(mm)
        yesses[mm] += mms_by_agent[condition][mm]["yes"]
        nos[mm] += mms_by_agent[condition][mm]["no"]
        print(np.mean(mms_by_agent[condition][mm]["yes"]) - np.mean(mms_by_agent[condition][mm]["no"]))
        print(ttest_ind(mms_by_agent[condition][mm]["yes"], mms_by_agent[condition][mm]["no"]))
    print('\n')
print("combined")
for mm in yesses:
    print(mm)
    print(ttest_ind(yesses[mm], nos[mm]))

faq
natural language
0.17361111111111427
Ttest_indResult(statistic=0.017442993171848796, pvalue=0.986265080470828)
keywords
-9.548611111111107
Ttest_indResult(statistic=-0.9834724604894658, pvalue=0.3377336517652062)
specific question
7.916666666666664
Ttest_indResult(statistic=0.7363805435005957, pvalue=0.47049414182339155)
follow-up questions
5.698529411764703
Ttest_indResult(statistic=0.4567905352640715, pvalue=0.6530008124860293)
general answer
0.7386363636363669
Ttest_indResult(statistic=0.07490646177507292, pvalue=0.9410721945216679)
personalized answer
-11.488970588235297
Ttest_indResult(statistic=-0.9368495160384853, pvalue=0.360592543013902)
immediate answer
-8.95432692307692
Ttest_indResult(statistic=-0.901515325752155, pvalue=0.3786007161084701)
long dialog
-1.9318181818181799
Ttest_indResult(statistic=-0.19607836959220276, pvalue=0.8466307479766229)


cts_llm
natural language
-5.88942307692308
Ttest_indResult(statistic=-0.5539585591256901, pvalue=0.5860685072290366)
keyword

/var/folders/xn/1drxn9fx2dxc6_bh03c675ym0000gq/T/ipykernel_69529/1340168586.py:28: RuntimeWarning:

Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.



### Role of Mental Models on Reliability

In [258]:
from scipy.stats import ttest_ind
import numpy as np
mms_by_agent = {}

for user in user_mms:
    condition = user_condition_mapping[user]
    if user in user_black_list:
        continue
    if condition not in mms_by_agent:
        mms_by_agent[condition] = {}
    for mm in user_mms[user]:
        if mm not in mms_by_agent[condition]:
            mms_by_agent[condition][mm] = {"yes": [], "no": []}
        if user_mms[user][mm] > 3:
            mms_by_agent[condition][mm]["yes"].append(u_reliability[user])
        elif user_mms[user][mm]<= 3:
            mms_by_agent[condition][mm]["no"].append(u_reliability[user])
    
yesses = defaultdict(lambda: list())
nos = defaultdict(lambda: list())
for condition in mms_by_agent:
    print(condition)
    for mm in mms_by_agent[condition]:
        print(mm)
        yesses[mm] += mms_by_agent[condition][mm]["yes"]
        nos[mm] += mms_by_agent[condition][mm]["no"]
        print(np.mean(mms_by_agent[condition][mm]["yes"]), np.mean(mms_by_agent[condition][mm]["no"]))
        print(ttest_ind(mms_by_agent[condition][mm]["yes"], mms_by_agent[condition][mm]["no"]))
    print('\n')
print("combined")
for mm in yesses:
    print(mm)
    print(ttest_ind(yesses[mm], nos[mm]))

faq
natural language
2.7916666666666665 2.7962962962962963
Ttest_indResult(statistic=-0.014435735447514388, pvalue=0.9886328569080743)
keywords
2.611111111111111 3.037037037037037
Ttest_indResult(statistic=-1.3943775012242212, pvalue=0.17929575435416387)
specific question
2.8444444444444446 2.6666666666666665
Ttest_indResult(statistic=0.5094758206486399, pvalue=0.6162852044553645)
follow-up questions
2.803921568627451 2.75
Ttest_indResult(statistic=0.13347468194489637, pvalue=0.8952215757663529)
general answer
2.7833333333333337 2.8030303030303028
Ttest_indResult(statistic=-0.06198949217549888, pvalue=0.9512186649005889)
personalized answer
2.5833333333333335 2.843137254901961
Ttest_indResult(statistic=-0.6499096970647064, pvalue=0.5235313405841906)
immediate answer
2.807692307692307 2.770833333333333
Ttest_indResult(statistic=0.11281938858566601, pvalue=0.9113572331152408)
long dialog
2.772727272727273 2.8166666666666664
Ttest_indResult(statistic=-0.13833988956920898, pvalue=0.8914274

/var/folders/xn/1drxn9fx2dxc6_bh03c675ym0000gq/T/ipykernel_69529/421500329.py:28: RuntimeWarning:

Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.



### Effect of mental models on trust

In [259]:
from scipy.stats import ttest_ind
import numpy as np
mms_by_agent = {}

for user in user_mms:
    condition = user_condition_mapping[user]
    if user in user_black_list:
        continue
    if condition not in mms_by_agent:
        mms_by_agent[condition] = {}
    for mm in user_mms[user]:
        if mm not in mms_by_agent[condition]:
            mms_by_agent[condition][mm] = {"yes": [], "no": []}
        if user_mms[user][mm] > 3:
            mms_by_agent[condition][mm]["yes"].append(u_trust[user])
        elif user_mms[user][mm]<= 3:
            mms_by_agent[condition][mm]["no"].append(u_trust[user])
    
yesses = defaultdict(lambda: list())
nos = defaultdict(lambda: list())
for condition in mms_by_agent:
    print(condition)
    for mm in mms_by_agent[condition]:
        print(mm)
        yesses[mm] += mms_by_agent[condition][mm]["yes"]
        nos[mm] += mms_by_agent[condition][mm]["no"]
        print(np.mean(mms_by_agent[condition][mm]["yes"]), np.mean(mms_by_agent[condition][mm]["no"]))
        print(ttest_ind(mms_by_agent[condition][mm]["yes"], mms_by_agent[condition][mm]["no"]))
    print('\n')
print("combined")
for mm in yesses:
    print(mm)
    print(ttest_ind(yesses[mm], nos[mm]))

faq
natural language
2.9583333333333335 2.6666666666666665
Ttest_indResult(statistic=0.7388014844184695, pvalue=0.4690566515723348)
keywords
2.75 2.9444444444444446
Ttest_indResult(statistic=-0.4886503598148862, pvalue=0.630681704714495)
specific question
3.033333333333333 2.3333333333333335
Ttest_indResult(statistic=1.7149437136629118, pvalue=0.10262018125413612)
follow-up questions
2.7941176470588234 3.0
Ttest_indResult(statistic=-0.4097917677666222, pvalue=0.6865420859921765)
general answer
2.95 2.727272727272727
Ttest_indResult(statistic=0.5660833954301252, pvalue=0.5779626154018704)
personalized answer
2.75 2.8529411764705883
Ttest_indResult(statistic=-0.2042201359004077, pvalue=0.8403530753007313)
immediate answer
2.8076923076923075 2.875
Ttest_indResult(statistic=-0.16507124113133015, pvalue=0.8706315301494496)
long dialog
2.8181818181818183 2.85
Ttest_indResult(statistic=-0.08020917846944609, pvalue=0.9369097211658846)


cts_llm
natural language
3.269230769230769 3.5625
Ttest_i

/var/folders/xn/1drxn9fx2dxc6_bh03c675ym0000gq/T/ipykernel_69529/1324597439.py:28: RuntimeWarning:

Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.



In [139]:
post_mental_models = {
      "natural language": [],
      "keywords": [],
      "specific question": [],
      "follow-up questions": [],
      "general answer": [],
      "personalized answer": [],
      "immediate answer": [],
      "long dialog": []
}

post_user_mms = {}

for res in post_surveys:
    user = res["user"]
    if user in user_black_list:
            continue
    if res["chat_exp_1"] != "None":
        post_mental_models["natural language"].append(int(res["chat_exp_1"]))
    if res["chat_exp_2"] != "None":
        post_mental_models["keywords"].append(int(res["chat_exp_2"]))
    if res["chat_exp_3"] != "None":
        post_mental_models["specific question"].append(int(res["chat_exp_3"]))
    if res["chat_exp_4"] != "None":
        post_mental_models["follow-up questions"].append(int(res['chat_exp_4']))
    if res["chat_exp_5"] != "None":
        post_mental_models["general answer"].append(int(res["chat_exp_5"]))
    if res["chat_exp_6"] != "None":
        post_mental_models["personalized answer"].append(int(res["chat_exp_6"]))
    if res["chat_exp_7"] != "None":
        post_mental_models["immediate answer"].append(int(res['chat_exp_7']))
    if res["chat_exp_8"] != "None":
        post_mental_models["long dialog"].append(int(res["chat_exp_8"]))
    post_user_mms[user] = {key: post_mental_models[key][-1] for key in post_mental_models}
    
labels = [key for key in post_mental_models]
avg_post_mental_models = [np.mean(post_mental_models[l]) for l in labels]

In [140]:
differences_by_condition = {}
before_by_condition = {}
after_by_condition = {}
for user in user_mms:
    if user in user_black_list or user == '80dce4aa2bb6e4747dea90c91ca9fe':
        continue
    condition = user_condition_mapping[user]
    if condition not in differences_by_condition:
        differences_by_condition[condition] = {}
    if condition not in before_by_condition:
        before_by_condition[condition] = {}
    if condition not in after_by_condition:
        after_by_condition[condition] = {}
    for mm in user_mms[user]:
        if mm not in differences_by_condition[condition]:
            differences_by_condition[condition][mm] = []
        if mm not in before_by_condition[condition]:
            before_by_condition[condition][mm] = []
        if mm not in after_by_condition[condition]:
            after_by_condition[condition][mm] = []
        diff = post_user_mms[user][mm] - user_mms[user][mm]
        differences_by_condition[condition][mm].append(diff)
        before_by_condition[condition][mm].append(user_mms[user][mm])
        after_by_condition[condition][mm].append(post_user_mms[user][mm])

for condition in differences_by_condition:
    print(condition)
    for mm in differences_by_condition[condition]:
        print(mm)
        print(ttest_ind(before_by_condition[condition][mm], after_by_condition[condition][mm]))
        print(np.mean(before_by_condition[condition][mm]), np.mean(after_by_condition[condition][mm]), np.mean(differences_by_condition[condition][mm]))

faq
natural language
Ttest_indResult(statistic=-0.13483997249264917, pvalue=0.8934146361709527)
3.4285714285714284 3.4761904761904763 0.047619047619047616
keywords
Ttest_indResult(statistic=0.1255900898258328, pvalue=0.9006855246323955)
3.5714285714285716 3.5238095238095237 -0.047619047619047616
specific question
Ttest_indResult(statistic=0.7989354619369607, pvalue=0.42904640807664196)
3.857142857142857 3.5714285714285716 -0.2857142857142857
follow-up questions
Ttest_indResult(statistic=9.339475705392294, pvalue=1.3342327927619451e-11)
4.190476190476191 1.7142857142857142 -2.4761904761904763
general answer
Ttest_indResult(statistic=-3.0745934690622105, pvalue=0.003788919094402054)
3.0476190476190474 4.095238095238095 1.0476190476190477
personalized answer
Ttest_indResult(statistic=1.8740851426632732, pvalue=0.06823600488768804)
2.4761904761904763 1.9047619047619047 -0.5714285714285714
immediate answer
Ttest_indResult(statistic=-1.2199885626608373, pvalue=0.22961349317512944)
3.61904761

In [55]:
for key in mental_models:
    print({f"{key}: {stats.ttest_ind(mental_models[key], post_mental_models[key])}"})

{'natural language: Ttest_indResult(statistic=0.28033098596050277, pvalue=0.7806683492514809)'}
{'keywords: Ttest_indResult(statistic=-0.9543482955111211, pvalue=0.34563950251914166)'}
{'specific question: Ttest_indResult(statistic=2.4237726026264967, pvalue=0.01997534226520323)'}
{'follow-up questions: Ttest_indResult(statistic=-1.294369603381147, pvalue=0.20296075493839177)'}
{'general answer: Ttest_indResult(statistic=-0.6412364700532214, pvalue=0.5250264189676689)'}
{'personalized answer: Ttest_indResult(statistic=-1.4509525002200236, pvalue=0.15459078143343824)'}
{'immediate answer: Ttest_indResult(statistic=0.4965635331614206, pvalue=0.6222149306594651)'}
{'long dialog: Ttest_indResult(statistic=-1.3286579139913106, pvalue=0.1916830191682154)'}


People thought that they needed to ask more general questions after interacting with the chatbot, but otherwise there were no significant changes in mental models

### Get Demographic Info

In [289]:
import csv
user_stats_per_policy = {}
user_exp_map = {}
exp_chat = []
exp_travel = []
gender = {}
with open('pre_survey.csv', 'r') as infile:
    reader = csv.DictReader(infile, delimiter="|")
    for row in reader:
        keep_keys = ['gender', 'age', 'experience_chatbots', 'experience_businesstravel', 'user']
        policy = user_condition_mapping[row['user']]
        if policy not in user_stats_per_policy:
            user_stats_per_policy[policy] = []
        exp = int(row['experience_chatbots'])
        user = row['user']
        if row["gender"] not in gender:
            gender[row["gender"]] = 0
        gender[row["gender"]] += 1
        user_exp_map[user] = exp
        user_stats_per_policy[policy].append({key: row[key] for key in keep_keys})
        exp_chat.append(int(row['experience_chatbots']))
        exp_travel.append(int(row['experience_businesstravel']))

print(np.mean(exp_chat), np.mean(exp_travel))
print(gender)

3.629032258064516 2.3870967741935485
{'male': 23, 'female': 39}


In [ ]:
for policy in user_stats_per_policy:
    print(policy)
    gender = {}
    
    for entry